# 3. Retrieve, check the evidence, and try again

The agent can search two collections: LangGraph documentation and LangChain documentation. We keep the original tutorial's corpus so the framework port can face the same questions.

Run `uv run llamaindex-rag setup --with-reranker` first. GoodMem stores, chunks and embeds the documents. LlamaIndex runs the workflow; the shared integration supplies its standard retriever.

The explicit loop is: **decide → retrieve → grade → draft or rewrite → decide**. A relevance check decides whether to draft from useful passages or suggest a better search. The original question and earlier evidence survive every step. Four searches is the hard limit.

In [1]:
from goodmem_rag.config import Settings, chat_model
from goodmem_rag.retrieval import make_tools

settings = Settings.from_env()
state = settings.state()
model = chat_model()

In [2]:
from goodmem_rag.agents import AgenticRAGWorkflow

async with settings.async_client() as client:
    tools = make_tools(async_client=client, state=state)
    print([(tool.metadata.name, tool.metadata.description) for tool in tools])
    workflow = AgenticRAGWorkflow(model, tools)
    result = await workflow.run(question="What is a checkpointer used for in LangGraph? Cite the docs.")

print(result.rendered())
print("Steps:", " → ".join(result.steps))
print("Searches:", result.calls)

[('langgraph_docs_tool', 'Search LangGraph docs for StateGraph, nodes, edges, reducers, persistence and workflows.'), ('langchain_docs_tool', 'Search LangChain docs for models, tools, prompts, built-in agents and tool-calling loops.')]


In LangGraph, a **checkpointer** is used to save and restore the state of a graph execution, particularly at super-step boundaries. It ensures that if execution is interrupted or needs to be resumed, the graph can continue from where it left off without re-executing completed tasks. Specifically, the checkpointer:

1. **Saves task results**: When a node contains multiple tasks, the checkpointer stores the results of completed tasks, allowing resumed threads to skip over already completed work.
2. **Restores completed task results**: On resume, LangGraph retrieves completed task results from the checkpointer, ensuring determinism and idempotency.
3. **Handles graph migrations**: The checkpointer supports migrations of graph definitions (nodes, edges, and state), enabling changes to the graph topology while preserving existing thread states.

**Citations**:
- [Graph API overview](https://docs.langchain.com/oss/python/langgraph/graph-api)

Sources consulted: [graph-api](https://docs.langc

## Ask a question that requires a dependent lookup

A multi-hop question needs information from one search to choose the next. Merely searching two collections in parallel is not enough for this example.

In [3]:
from goodmem_rag.evaluation import SEQUENTIAL_QUESTION

async with settings.async_client() as client:
    result = await AgenticRAGWorkflow(model, make_tools(async_client=client, state=state)).run(
        question=SEQUENTIAL_QUESTION)

print(result.rendered())
for call in result.calls:
    print(call["round"], call["name"], call["args"])

The LangGraph overview recommends using LangChain's [agents] for a higher-level abstraction when getting started with agents or wanting prebuilt architectures for common LLM and tool-calling loops.

LangChain's agents can be constructed using the `create_agent()` function. The tool-calling loop is handled by the agent loop, which executes the tools and passes the results back to the model, creating a conversation loop where the model can use tool results to generate its final response.

**Cited sources:**

* LangGraph overview: https://docs.langchain.com/oss/python/langgraph/overview
* LangChain agents: https://docs.langchain.com/oss/python/langchain/tools

Sources consulted: [overview](https://docs.langchain.com/oss/python/langgraph/overview), [workflows-agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents), [graph-api](https://docs.langchain.com/oss/python/langgraph/graph-api), [models](https://docs.langchain.com/oss/python/langchain/models), [tools](https://docs.

Open `goodmem_rag/agents.py` to follow the typed events and workflow steps. Change `max_searches` to see the budget affect the route. Source links come from stored metadata and remain available on `result.nodes`, independently of the model's answer.